In [43]:
%cd PRO505

[Errno 2] No such file or directory: 'PRO505'
/workspace/PRO505


In [44]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType, IntegerType, FloatType,BooleanType
from pyspark.sql import functions as fun
import pandas as pd 

spark = ( SparkSession.builder.appName("PRO505").master("spark://spark-master:7077").getOrCreate())



sc = spark.sparkContext

In [45]:
main_schema = StructType([
    StructField("index",StringType(),True),
    StructField("Title",StringType(),True),
    StructField("Description",StringType(),True),
    StructField("Amount(in rupees)",StringType(),True),
    StructField("Price(in rupees)",FloatType(),True),
    StructField("location",StringType(),True),
    StructField("Carpet Area",StringType(),True),
    StructField("Status",StringType(),True),
    StructField("Floor",StringType(),True),
    StructField("Transaction",StringType(),True),
    StructField("Furnishing",StringType(),True),
    StructField("facing",StringType(),True),
    StructField("overlooking",StringType(),True),
    StructField("Society",StringType(),True),
    StructField("Bathroom",IntegerType(),True),
    StructField("Balcony",IntegerType(),True),
    StructField("Car Parking",StringType(),True),
    StructField("Ownership",StringType(),True),
    StructField("Super Area",StringType(),True),
    StructField("Dimensions",FloatType(),True),
    StructField("Plot Area",FloatType(),True),
])

df = (spark.read.format("csv")
      .option("header","true")
      .schema(main_schema)
      .option("sep",";")
      .load("house_prices.csv"))

df.show(5)

+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+
|index|               Title|         Description|Amount(in rupees)|Price(in rupees)|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|
+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+
|    0|1 BHK Ready to Oc...|Bhiwandi, Thane h...|          42 Lac |          6000.0|   thane|   500 sqft|Ready to Move|10 out o

In [46]:
split_col = fun.split(fun.col("Amount(in rupees)"), " ")
 
df = df.withColumn(
    "Amount(in rupees)",
    fun.when(split_col.getItem(1) == "Lac",
           split_col.getItem(0).cast("double") * 100000)
     .otherwise(
           split_col.getItem(0).cast("double") * 10000000
     )
)

df = df.withColumn(
    "Amount_USD",
    (fun.col("Amount(in rupees)")*0.012)
)

df.show(5)
 

+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+
|index|               Title|         Description|Amount(in rupees)|Price(in rupees)|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|
+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+
|    0|1 BHK Ready to Oc...|Bhiwandi, Thane h...|        4200000.0|          6000.0|   thane| 

In [47]:
# df = df.withColumn(
# "Carpet Area",
# fun.regexp_replace(fun.col("Carpet Area"),"[^0-9]"," ").cast("float")
# )

df = df.withColumn(
    "Carpet Area",
    fun.split(fun.col("Carpet Area")," ").getItem(0).cast("float")
)

df = df.withColumn(
    "Area_m2",
    fun.col("Carpet Area")*0.0929
)

df.show(5)

+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+
|index|               Title|         Description|Amount(in rupees)|Price(in rupees)|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|
+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+
|    0|1 BHK Ready to Oc...|Bhiwandi,

In [48]:
resultado_df = df.select(fun.stddev("Amount_USD"))
Amount_desviacion = resultado_df.collect()[0][0]

resultado_df = df.select(fun.variance("Amount_USD"))
Amount_varianza = resultado_df.collect()[0][0]

print(Amount_desviacion)
print(Amount_varianza)

473259.23319444305
223974301803.79224


En el caso de que la desviacion sea 80k y le madedia 100k no se deveria confiar en la media, devido a que estaria influida porque casa mas caras , lo mejor seria confiar en la mediana

In [49]:
resultado_df = df.select(fun.kurtosis("Amount_USD"))
Amount_kurtois = resultado_df.collect()[0][0]

resultado_df = df.select(fun.skewness("Amount_USD"))
Amount_skewness = resultado_df.collect()[0][0]

resultado_df = df.select(fun.median("Amount_USD"))
Amount_mediana = resultado_df.collect()[0][0]

resultado_df = df.select(fun.mean("Amount_USD"))
Amount_media = resultado_df.collect()[0][0]


print(Amount_kurtois)
print(Amount_skewness)
print(Amount_mediana)
print(Amount_media)

91251.1946216962
270.25366063569174
93600.0
143776.13026927641


En este caso existen una mayoria de casas baratas con algunas mansiones ultra-caras 

En este caso existen una serie de casas de lujo extremo que nos están distorsionando los datos

In [50]:
df = df.withColumn(
    "AmountNorm", 
    (fun.col("Amount_USD") - Amount_media) / Amount_desviacion 
)

df.show(5)

+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+--------------------+
|index|               Title|         Description|Amount(in rupees)|Price(in rupees)|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|          AmountNorm|
+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+---------------

In [51]:
resultado_df = df.select(fun.stddev("Area_m2"))
Area_desviacion = resultado_df.collect()[0][0]

resultado_df = df.select(fun.mean("Area_m2"))
Area_media = resultado_df.collect()[0][0]

df = df.withColumn(
    "AreaNorm", 
    (fun.col("Area_m2") - Area_media) / Area_desviacion 
)

df.show(5)

+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+--------------------+--------------------+
|index|               Title|         Description|Amount(in rupees)|Price(in rupees)|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|          AmountNorm|            AreaNorm|
+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+-----

In [52]:
Q_high = df.approxQuantile("Amount_USD", [0.99], 0.01)[0]

print(Q_high)

df_limpio = df.filter(fun.col("Amount_USD") <= Q_high)

df_limpio.show(5)

168036000.0
+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+--------------------+--------------------+
|index|               Title|         Description|Amount(in rupees)|Price(in rupees)|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|          AmountNorm|            AreaNorm|
+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----

In [53]:
resultado_df = df_limpio.select(fun.stddev("Amount_USD"))
Amount_desviacion_limpio = resultado_df.collect()[0][0]

resultado_df = df_limpio.select(fun.kurtosis("Amount_USD"))
Amount_kurtois_limpio = resultado_df.collect()[0][0]

resultado_df = df_limpio.select(fun.mean("Amount_USD"))
Amount_media_limpio = resultado_df.collect()[0][0]

print(f"Amount_desviacion_limpio {Amount_desviacion_limpio}")
print(f"Amount_kurtois_limpio {Amount_kurtois_limpio}")
print(f"Amount_media_limpio {Amount_media_limpio}")


Amount_desviacion_limpio 473259.23319444305
Amount_kurtois_limpio 91251.1946216962
Amount_media_limpio 143776.13026927641


El kurtois no baja absolutamente nada devido a que al eliminar solo el 1% de los datos no es suficiente para normalizar las metricas y no serian validas para entrenar ningun modelo 

In [54]:
df_limpio = df_limpio.withColumn(
    "Num_Bedrooms",
    fun.split(fun.col("Title")," ").getItem(0)
)

df_limpio = df_limpio.filter(fun.col("Num_Bedrooms").isNotNull())

df_limpio = df_limpio.withColumn(
    "Num_Bedrooms", 
    fun.regexp_extract(fun.col("Num_Bedrooms"), r"(\d+)", 1).cast("int")
)

df_limpio.show(5)

+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+--------------------+--------------------+------------+
|index|               Title|         Description|Amount(in rupees)|Price(in rupees)|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|          AmountNorm|            AreaNorm|Num_Bedrooms|
+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+-----------

In [55]:
stats = (df_limpio.groupBy("Num_Bedrooms")
         .agg(
             fun.mean("Amount_USD"),
             fun.stddev("Amount_USD"),
             fun.kurtosis("Amount_USD")
         ))

df_limpio = df_limpio.join(stats,"Num_Bedrooms")

stats.show()

df_limpio.show(5)

+------------+------------------+------------------+--------------------+
|Num_Bedrooms|   avg(Amount_USD)|stddev(Amount_USD)|kurtosis(Amount_USD)|
+------------+------------------+------------------+--------------------+
|        NULL|173057.63688760807| 686948.3095026173|  109.83179735566162|
|           1| 42374.56647398844|32281.365649823914|   33.01952265661249|
|           6| 844643.4782608695| 910736.4390346808|    3.23962180287596|
|           3| 165414.9269135928| 647172.2810475255|  58546.689824579495|
|           5|  556257.212971078|512576.11489171913|   68.28443827060599|
|           9|436285.71428571426| 190736.9167952849| -1.4406103915346804|
|           4|367763.79285014694| 294356.1288514866|    43.7083890185256|
|           8|         2507820.0|1946900.5176433644|  -1.799848731210454|
|           7| 682742.8571428572| 725893.0310806328|   2.679962529186576|
|          10| 796254.5454545454|1363400.5166227834|   5.207930779470404|
|           2| 74568.32623871956|21263

+------------+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+---------------+-----------+-----------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+--------------------+--------------------+-----------------+------------------+--------------------+
|Num_Bedrooms|index|               Title|         Description|Amount(in rupees)|Price(in rupees)|location|Carpet Area|       Status|          Floor|Transaction| Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|          AmountNorm|            AreaNorm|  avg(Amount_USD)|stddev(Amount_USD)|kurtosis(Amount_USD)|
+------------+-----+--------------------+--------------------+-----------------+----------------+--------+-----------+-------------+

El grupo de 3 habitaciones es el mas heterogeneo. El grupo de una habitacion actua de forma mas homogenea, es decir sus precios son mas similares entre si mientras que las viviendas de 3 haitaciones hay una gran diferencia de precios devido a que engloba viviendas mas económicas con viviendas de lujo

El precio promedio es mas fiable en las viviendas de 1 habitacion. En las viviendas de 3 habitaciones hay mayor riesgo de equivocarse debido a que la desviacion es mucho mayor

El segmento de 3 BHK tiene una curtosis abismalmente más alta. Esto indica que el segmento de 3 habitaciones tiene colas mas pesadas, lo que significa que la mayoría de las biviendas siguen un precio estándar, pero existen unos pocos valores mas elevados que disparan la métrica y distorsionan toda la distribución.

Un kurtois tan extremo indica que las biviendas de lujo son una anomalia estadìstica , dando un falso valor estandar para las biviendas de 3 habitaciones. Lo mas optimo para entrenar un modelo seria tratarlas como un grupo aparte de biviendas de lujo agrupandolas con otras viviendas de luho sin importar su numero de habitaciones 